# 파일 불러오기

In [ ]:
# !pip install pypdf

In [125]:
import json
import re
import time
from pathlib import Path
from typing import List, Dict, Any

from openai import OpenAI
from pypdf import PdfReader

In [126]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [127]:
# 1. 기본 설정
MODEL_NAME = "gpt-4.1-mini"

INPUT_FILES = [
    "./data/fia_2026_f1_regulations_-_section_d_financial_-_f1_teams_-_iss_05_-_2026-02-27.md",
    "./data/fia_2026_f1_regulations_-_section_f_operational_-_iss_06_-_2026-02-27.md",
]

OUTPUT_DIR = Path("output_qa_pipeline")
OUTPUT_DIR.mkdir(exist_ok=True)

FINETUNE_JSONL_PATH = OUTPUT_DIR / "f1_finetune_qa.jsonl"

SLEEP_SECONDS = 0.5
MAX_RETRIES = 3

client = OpenAI()

In [128]:
def read_markdown(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_multiple_markdowns(paths: List[str]) -> List[Dict[str, str]]:
    docs = []
    for path in paths:
        text = read_markdown(path)
        docs.append({
            "source_file": Path(path).name,
            "text": text
        })
    return docs

In [ ]:
# 3. 텍스트 정리 + 문장 분리
def normalize_markdown(text: str) -> str:
    # 페이지 푸터 제거: "> 27 February 2026 D4 Issue 05"
    text = re.sub(r'>\s*\d+ \w+ \d+ [A-Z]\d+ Issue \d+\s*\n', '', text)
    # 페이지 헤더 제거: "SECTION D: FINANCIAL REGULATIONS..."
    text = re.sub(r'SECTION [A-Z]: .+?\n', '', text)
    # **볼드** 제거
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    # _이탤릭_ 제거
    text = re.sub(r'_(.+?)_', r'\1', text)
    # ## 헤더 기호 제거
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def split_articles(text: str, source_file: str) -> List[Dict[str, Any]]:
    text = normalize_text(text)

    # 수정 1: 앞 공백 허용 + 점 없는 상위 섹션도 포함
    raw_header_pattern = re.compile(
        r'^(#{1,3})\s+\**\s*([A-Z]\d+(?:\.\d+){0,3})[^\n]*?\**\s*$',
        re.MULTILINE
    )
    matches = list(pattern.finditer(text))

    articles = []

    for i, match in enumerate(matches):
        article_id = match.group(1).strip()
        article_title = match.group(2).strip()

        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        article_text = text[start:end].strip()

        # 수정 2: 너무 짧은 것 제외 (기존)
        if len(article_text) < 100:
            continue

        # 수정 3: [Replaced by...] 단독 조문 제외
        body = article_text[len(match.group(0)):].strip()
        if re.match(r'^\[Replaced by', body) and len(body) < 80:
            continue

        # 수정 4: 목차 제외 (한 조문 안에 조문 번호가 5개 이상 나열된 경우)
        sub_article_count = len(re.findall(r'[A-Z]\d+\.\d+', article_text))
        if sub_article_count > 8 and '\n' not in article_text[:200]:
            continue

        articles.append({
            "source_file": source_file,
            "article_id": article_id,
            "article_title": article_title,
            "article_text": article_text
        })

    return articles

In [130]:
def split_articles(text: str, source_file: str) -> List[Dict[str, Any]]:
    raw_header_pattern = re.compile(
        r'^(#{1,3})\s+\**\s*([A-Z]\d+(?:\.\d+){0,3})[^\n]*?\**\s*$',
        re.MULTILINE
    )

    matches = list(raw_header_pattern.finditer(text))  # ✅ pattern → raw_header_pattern

    articles = []

    for i, match in enumerate(matches):
        article_id = match.group(2).strip()

        header_line = match.group(0)
        title_part = re.sub(r'^#{1,3}\s+', '', header_line)
        title_part = re.sub(r'\*\*?', '', title_part).strip()
        article_title = re.sub(
            r'^[A-Z]\d+(?:\.\d+){0,3}\s*', '', title_part
        ).strip()

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        raw_body = text[start:end]

        body = normalize_markdown(raw_body).strip()
        article_text = f"{article_id} {article_title}\n\n{body}".strip()

        if len(article_text) < 100:
            continue

        if re.search(r'\[Replaced by', body) and len(
            re.sub(r'\[Replaced by[^\]]+\]', '', body).strip()
        ) < 50:
            continue

        if re.match(r'^(VOID|void)\s*$', body.strip()):
            continue

        articles.append({
            "source_file": source_file,
            "article_id": article_id,
            "article_title": article_title if article_title else article_id,
            "article_text": article_text
        })

    return articles

In [ ]:
def split_definitions(text: str, source_file: str) -> List[Dict[str, Any]]:
    """Appendix D1 정의 섹션 전용 파서"""
    # ** 볼드 제거 먼저
    clean = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    # 페이지 푸터 제거
    clean = re.sub(r'>\s*\d+ \w+ \d+ [A-Z]\d+ Issue \d+\s*\n', '', clean)
    clean = re.sub(r'SECTION [A-Z]: .+?\n', '', clean)

    pattern = re.compile(
        r'"\s*([^"]+?)\s*"\s+means?\s+(.+?)(?=\n"\s*[^"]+?\s*"\s+means?|\Z)',
        re.DOTALL
    )

    articles = []
    for match in pattern.finditer(clean):  # text → clean
        term = match.group(1).strip()
        definition = match.group(2).strip()

        if len(definition) < 30:
            continue

        # continue가 for 루프 안에 있어야 함
        if re.match(r'^has the meaning set out in Article', definition) and len(definition) < 80:
            continue

        articles.append({
            "source_file": source_file,
            "article_id": f"DEF.{re.sub(r'[^A-Za-z0-9]', '_', term)}",
            "article_title": f"Definition: {term}",
            "article_text": f'"{term}" means {definition}'
        })

    return articles

In [132]:
# 4. Q & A 생성
def build_prompt(article: Dict[str, Any]) -> str:
    return f"""
너는 F1 입문자용 규정 설명 챗봇의 파인튜닝 데이터를 만드는 도우미야.

아래 규정 조문만 보고 한국어 질문-답변 데이터를 만들어.
규정에 없는 내용은 추측하지 마.
숫자, 조건, 예외는 원문 기준으로 유지해.

[조문 정보]
- 파일명: {article["source_file"]}
- 조문 ID: {article["article_id"]}
- 조문 제목: {article["article_title"]}

[조문 원문]
{article["article_text"]}

[생성 규칙]
- 조문 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 6  문장
- 숫자, 단위(mm, ms, 개수, 각도)는 조문 기준 그대로 유지
- 규정 원문에 없는 내용 단정 금지
- 질문끼리 의미 중복이 크지 않게 작성
- 오해 교정형은 왜 틀렸는지도 설명할 것
- 비교형은 같은 조문 안에서 비교 가능한 대상이 있을 때만 만들 것
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것
- 조항 번호만 언급하지 말고, 내용을 풀어서 설명할 것
- 아래 유형을 고루 섞기:
1. 개념형
2. 요약형
3. 수치/기준형
4. 오해 교정형

[출력 형식]
반드시 아래 JSON 배열만 출력:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]
""".strip()


def extract_json_array(text: str) -> str:
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)


def generate_qa(article: Dict[str, Any]) -> List[Dict[str, Any]]:
    prompt = build_prompt(article)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )

            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                data = json.loads(extract_json_array(text))

            results = []
            for item in data:
                if not isinstance(item, dict):
                    continue
                if "question" not in item or "answer" not in item:
                    continue

                results.append({
                    "source_file": article["source_file"],
                    "article_id": article["article_id"],
                    "article_title": article["article_title"],
                    "question": item["question"],
                    "answer": item["answer"]
                })

            return results

        except Exception as e:
            print(f"[재시도 {attempt + 1}/{MAX_RETRIES}] {article['article_id']} 실패: {e}")
            time.sleep(1.5)

    return []

In [133]:
# 5. 중복 제거
def normalize_question(q: str) -> str:
    q = q.strip().lower()
    q = re.sub(r"[^\w가-힣\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q


def deduplicate_qa(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in data:
        key = (
            normalize_question(item["question"]),
            re.sub(r"\s+", " ", item["answer"].strip().lower())
        )

        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    return deduped

In [134]:
def to_finetune_messages(item):
    return {
        "messages": [
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }

---

In [135]:
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

---

In [136]:
def main():
    print("[1] 마크다운 파일 불러오기")
    docs = load_multiple_markdowns(INPUT_FILES)

    print("[2] 조문 분리")
    articles = []
    for doc in docs:
        print(f"  - 처리 중: {doc['source_file']}")

        # 기본 조문 분리
        raw = split_articles(doc["text"], doc["source_file"])

        # 긴 조문은 하위 항목별로 추가 분리
        chunked = []
        for a in raw:
            chunked.extend(split_long_article(a, max_chars=3000))

        # Definitions 섹션 별도 파싱
        defs = split_definitions(doc["text"], doc["source_file"])

        print(f"    → 조문: {len(raw)}개, 청킹 후: {len(chunked)}개, 정의: {len(defs)}개")

        articles.extend(chunked)
        articles.extend(defs)

    print(f"\n총 조문 수: {len(articles)}")

    print("\n[3] QA 생성")
    all_qa = []
    for i, article in enumerate(articles, start=1):
        print(f"  - ({i}/{len(articles)}) [{article['article_id']}] {article['article_title'][:40]}")
        qa_items = generate_qa(article)
        all_qa.extend(qa_items)
        time.sleep(SLEEP_SECONDS)

    print(f"\n생성된 QA 수: {len(all_qa)}")

    print("[4] 중복 제거")
    deduped = deduplicate_qa(all_qa)
    print(f"중복 제거 후 QA 수: {len(deduped)}")

    print("[5] JSONL 저장")
    rows = [to_finetune_messages(item) for item in deduped]
    save_jsonl(FINETUNE_JSONL_PATH, rows)

    print("\n완료")
    print(f"저장 위치: {FINETUNE_JSONL_PATH}")



if __name__ == "__main__":
    main()

[1] 마크다운 파일 불러오기
[2] 조문 분리
  - 처리 중: fia_2026_f1_regulations_-_section_d_financial_-_f1_teams_-_iss_05_-_2026-02-27.md
    → 조문: 32개, 청킹 후: 32개, 정의: 109개
  - 처리 중: fia_2026_f1_regulations_-_section_f_operational_-_iss_06_-_2026-02-27.md
    → 조문: 11개, 청킹 후: 11개, 정의: 0개

총 조문 수: 152

[3] QA 생성
  - (1/152) [D1.1] Scope
  - (2/152) [D1.3] Interpretation
  - (3/152) [D1.4] Transitional provisions
  - (4/152) [D2.1] Obligations of F1 Teams
  - (5/152) [D3.1] Obligations of individual F1 Team member
  - (6/152) [D4.1] Compliance with the Cost Cap
  - (7/152) [D4.2] Reporting Group
  - (8/152) [D5.1] Exclusions
  - (9/152) [D6.1.m] Adjustments (m)
  - (10/152) [D7.1] Full Year Reporting Documentation
  - (11/152) [D7.2] Interim Reporting Documentation
  - (12/152) [D8.1] General
  - (13/152) [D8.3] Ongoing compliance monitoring
  - (14/152) [D8.4] Requests for information/access and Dema
  - (15/152) [D9.1] Cost Cap Adjudication Panel
  - (16/152) [D9.2] Referral to the Cost Cap Adjudication 